# M2: Perbandingan strategi penyejajaran dan fusi
IF25-40304 · Kelompok 09 · vegapunk

**Pendekatan utama: (b) prompt ensemble berbahasa Indonesia** di atas encoder beku (tanpa pelatihan, CPU).
Varian: **V0** baseline M1 · **V1** ensemble templat Indonesia netral · **V2a** ensemble templat budaya · **V2b** terjemahan Inggris (NLLB-200 distilled 600M) · **V2** budaya + terjemahan.

**Cara pakai:** taruh notebook ini di folder yang sama dengan notebook M1 (supaya `cache/` M1 terbaca), atur `DIR` di sel kode pertama, lalu **Run All**. Pertama kali jalan, model NLLB (±2,5 GB) diunduh dari Hugging Face.

**Aturan leakage:** templat dan semua pengaturan ditetapkan *sebelum* melihat skor val dan dikunci di `hasil/m2_templat_terkunci.json`. Terjemahan hanya memakai `caption_id`. Kolom `category` dan `caption_en` bukan masukan varian mana pun. Tidak ada yang disetel dengan melihat val.

Atribusi data: SEA-VL Indonesian subset (KORIKA-AI/sea-vl_crowdsourcing_id), diturunkan dari SEACrowd/sea-vl_crowdsourcing, lisensi CC BY-SA 4.0 (creativecommons.org/licenses/by-sa/4.0). Cahyawijaya et al. (2025), *Crowdsource, Crawl, or Generate? Creating SEA-VL, a Multicultural Vision-Language Dataset for Southeast Asia*, ACL.

In [ ]:
import os, re, json, time, random, platform, sys, hashlib, datetime
import numpy as np, pandas as pd, torch
from PIL import Image
from IPython.display import display
import sentence_transformers
from sentence_transformers import SentenceTransformer

# ---------- PENGATURAN (sama dengan M1 bila ada) ----------
DIR = "../bundle-mahasiswa"   # folder berisi train.csv, val.csv, images/
SEED = 42
K_MAP = 10                    # K untuk mAP@K
IMG_MODEL = "sentence-transformers/clip-ViT-B-32"                  # encoder gambar + teks Inggris
TXT_MODEL = "sentence-transformers/clip-ViT-B-32-multilingual-v1"  # encoder teks Indonesia
MT_MODEL  = "facebook/nllb-200-distilled-600M"                     # penerjemah (hanya inferensi)
BATCH_IMG, BATCH_TXT, BATCH_MT = 32, 64, 16
N_BOOT = 1000                 # resample bootstrap berpasangan
# ----------------------------------------------------------

OUT = os.path.abspath(".")    # cache/ dan hasil/ di folder notebook (sama dengan M1)
os.chdir(DIR)
os.makedirs(f"{OUT}/cache", exist_ok=True); os.makedirs(f"{OUT}/hasil", exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

T_PATH, M_PATH = f"{OUT}/cache/waktu_m2.json", f"{OUT}/cache/m2_meta.json"
TIMES = json.load(open(T_PATH)) if os.path.exists(T_PATH) else {}
META  = json.load(open(M_PATH)) if os.path.exists(M_PATH) else {}
T1    = json.load(open(f"{OUT}/cache/waktu.json")) if os.path.exists(f"{OUT}/cache/waktu.json") else {}   # waktu M1

# [MODEL-LOAD]
img_model = SentenceTransformer(IMG_MODEL, device="cpu")
txt_model = SentenceTransformer(TXT_MODEL, device="cpu")
print("model dimuat:", IMG_MODEL, "|", TXT_MODEL)

## Templat dan konfigurasi (dikunci sebelum menjalankan)
Dua set templat dengan jumlah sama (K = 6) supaya V1 vs V2a mengukur *isi* templat (netral vs budaya), bukan jumlah ensemble.
`{c}` diganti caption Indonesia. Kolom `category` dari data **tidak** dipakai. Jangan ubah sel ini setelah melihat skor val; kalau hash berubah, notebook berhenti.

In [ ]:
NETRAL = [                                   # V1: tanpa konteks budaya
    "Foto: {c}",
    "Sebuah foto. {c}",
    "Gambar ini memperlihatkan: {c}",
    "Foto dengan keterangan: {c}",
    "{c} (foto)",
    "Foto yang menggambarkan hal berikut. {c}",
]
BUDAYA = [                                   # V2a: konteks budaya Indonesia, tanpa kategori per item
    "Foto budaya Indonesia: {c}",
    "Dokumentasi tradisi dan budaya nusantara: {c}",
    "Foto yang diambil di Indonesia. {c}",
    "Foto warisan budaya Indonesia dengan keterangan: {c}",
    "Budaya Indonesia: {c}",
    "Foto tentang adat, seni, dan kuliner khas Indonesia. {c}",
]
TERJEMAHAN = {"model": MT_MODEL, "src": "ind_Latn", "tgt": "eng_Latn", "num_beams": 4, "max_new_tokens": 160}
BOBOT_V2 = {"A_budaya": 1.0, "B_terjemahan": 1.0}     # V2 = L2( 1.0*A + 1.0*B ), tidak disetel
assert all(t.count("{c}") == 1 for t in NETRAL + BUDAYA) and len(NETRAL) == len(BUDAYA)

KONFIG = {"NETRAL": NETRAL, "BUDAYA": BUDAYA, "TERJEMAHAN": TERJEMAHAN, "BOBOT_V2": BOBOT_V2, "seed": SEED,
          "IMG_MODEL": IMG_MODEL, "TXT_MODEL": TXT_MODEL}
SHA = hashlib.sha256(json.dumps(KONFIG, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()
LOCK = f"{OUT}/hasil/m2_templat_terkunci.json"
if os.path.exists(LOCK):
    lama = json.load(open(LOCK))
    assert lama["sha256"] == SHA, "KONFIGURASI BERUBAH setelah dikunci pada " + lama["waktu_kunci"] + ". Kembalikan, atau catat alasannya di laporan."
    print("Konfigurasi identik dengan yang dikunci pada", lama["waktu_kunci"], "| sha256:", SHA[:12])
else:
    json.dump({"sha256": SHA, "waktu_kunci": datetime.datetime.now().isoformat(timespec="seconds"), "konfig": KONFIG},
              open(LOCK, "w"), ensure_ascii=False, indent=1)
    print("Konfigurasi dikunci:", SHA[:12], "(commit file notebook ini ke git SEBELUM melihat skor)")

## Data dan baseline M1 (dari cache)
V0 dibaca dari cache M1 (`val_img`, `val_txt_id`, `val_txt_en`), jadi angkanya identik dengan M1. Jika cache belum ada, dihitung ulang.

In [ ]:
tr = pd.read_csv("train.csv"); va = pd.read_csv("val.csv")
N = len(va); cap_id = va["caption_id"].astype(str).tolist()
print("train:", len(tr), "| val:", N)

def enc_images(files, bs=BATCH_IMG):
    out = []
    for i in range(0, len(files), bs):
        ims = [Image.open(f).convert("RGB") for f in files[i:i + bs]]
        out.append(img_model.encode(ims, batch_size=bs, normalize_embeddings=True))
        for im in ims: im.close()
    return np.concatenate(out)

def cached(name, fn, tkey=None):
    p = f"{OUT}/cache/{name}.npy"
    if os.path.exists(p): return np.load(p)
    t0 = time.time(); x = fn(); TIMES[tkey or name] = round(time.time() - t0, 1)
    np.save(p, x); json.dump(TIMES, open(T_PATH, "w"), indent=1)
    print(f"{name}: {x.shape} dalam {TIMES[tkey or name]} dtk"); return x

Ev  = cached("val_img",    lambda: enc_images(list(va["file"])))
Tiv = cached("val_txt_id", lambda: txt_model.encode(cap_id, batch_size=BATCH_TXT, normalize_embeddings=True))
Tev = cached("val_txt_en", lambda: img_model.encode(list(va["caption_en"]), batch_size=BATCH_TXT, normalize_embeddings=True))

def ranks_of_truth(S, tol=1e-6):
    d = np.diag(S)[:, None]
    return (S >= d - tol).sum(1)           # peringkat jawaban benar, tie pesimistik (sama dengan M1)

def n_tie(S, tol=1e-6):
    d = np.diag(S)[:, None]
    return int(((np.abs(S - d) <= tol).sum(1) > 1).sum())

def metrics(S, ks=(1, 5, 10), K=K_MAP):
    r = ranks_of_truth(S)
    out = {f"R@{k}": float((r <= k).mean()) for k in ks}
    out[f"mAP@{K}"] = float(np.where(r <= K, 1.0 / r, 0.0).mean())
    return out, r

ARAH = {
    "gambar → teks ID": lambda E, Ti, Te: E @ Ti.T,
    "gambar → teks EN": lambda E, Ti, Te: E @ Te.T,
    "teks ID → gambar": lambda E, Ti, Te: Ti @ E.T,
    "teks EN → gambar": lambda E, Ti, Te: Te @ E.T,
}
ARAH_ID = ["teks ID → gambar", "gambar → teks ID"]    # arah yang dipengaruhi varian (sisi Indonesia)

## Terjemahan NLLB (komponen B)
Caption Indonesia val → Inggris dengan NLLB-200 distilled 600M (beam 4, deterministik, hanya inferensi). Masukan hanya `caption_id`; `caption_en` (terjemahan bawaan data) **tidak** dipakai. Hasil disimpan agar tidak diterjemahkan ulang.

In [ ]:
TR_CSV = f"{OUT}/hasil/m2_terjemahan_val.csv"

# [NLLB-LOAD]
def terjemahkan(teks):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang=TERJEMAHAN["src"])
    mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL).eval()
    bos = tok.convert_tokens_to_ids(TERJEMAHAN["tgt"])
    urut = np.argsort([len(t) for t in teks]); hasil = [None] * len(teks)
    for i in range(0, len(teks), BATCH_MT):
        ids = urut[i:i + BATCH_MT]
        enc = tok([teks[j] for j in ids], return_tensors="pt", padding=True, truncation=True, max_length=200)
        with torch.no_grad():
            out = mt.generate(**enc, forced_bos_token_id=bos, num_beams=TERJEMAHAN["num_beams"],
                              max_new_tokens=TERJEMAHAN["max_new_tokens"])
        for j, s in zip(ids, tok.batch_decode(out, skip_special_tokens=True)): hasil[j] = s
    return hasil, sum(p.numel() for p in mt.parameters())

ok = False
if os.path.exists(TR_CSV):
    d = pd.read_csv(TR_CSV, keep_default_na=False)
    ok = (len(d) == N) and (d["caption_id"].astype(str).tolist() == cap_id)
if ok:
    trans = d["terjemahan_en"].astype(str).tolist(); print("terjemahan dibaca dari", TR_CSV)
else:
    t0 = time.time(); trans, META["mt_params"] = terjemahkan(cap_id)
    TIMES["t_mt_val"] = round(time.time() - t0, 1); META["mt_model"] = MT_MODEL
    pd.DataFrame({"caption_id": cap_id, "terjemahan_en": trans}).to_csv(TR_CSV, index=False)
    json.dump(TIMES, open(T_PATH, "w"), indent=1); json.dump(META, open(M_PATH, "w"), indent=1)
    print(f"{N} caption diterjemahkan dalam {TIMES['t_mt_val']} dtk")
display(pd.DataFrame({"caption_id": cap_id[:5], "terjemahan_en": trans[:5]}))

## Membangun embedding tiap varian
Tiap templat dienkode lalu dinormalisasi L2, dirata-rata, dinormalisasi ulang (sesuai proposal M0).
Hanya sisi **Indonesia** yang berubah; arah Inggris tetap baseline M1. Komponen: **A** = ensemble templat budaya, **B** = terjemahan Inggris (encoder CLIP asli).
V0 = caption polos · V1 = ensemble netral · V2a = A · V2b = B · V2 = L2(A + B).

In [ ]:
nrm = lambda x: x / np.linalg.norm(x, axis=-1, keepdims=True)
enc_id = lambda xs: txt_model.encode(xs, batch_size=BATCH_TXT, normalize_embeddings=True)
enc_en = lambda xs: img_model.encode(xs, batch_size=BATCH_TXT, normalize_embeddings=True)

def per_templat(templat, nama):                       # hasil: (K, N, D); cache bernama sesuai hash konfigurasi
    return cached(f"m2_{nama}_{SHA[:8]}",
                  lambda: np.stack([enc_id([t.replace("{c}", c) for c in cap_id]) for t in templat]),
                  tkey=f"t_{nama}")

PT_N = per_templat(NETRAL, "netral")                  # (6, N, D)
PT_C = per_templat(BUDAYA, "budaya")
H_T = hashlib.md5("\n".join(trans).encode("utf-8")).hexdigest()[:8]
B_T = cached(f"m2_terjemahan_enc_{H_T}", lambda: enc_en(trans), tkey="t_terj_enc")

A_N, A_C = nrm(PT_N.mean(0)), nrm(PT_C.mean(0))
VAR = {"V0": Tiv, "V1": A_N, "V2a": A_C, "V2b": B_T,
       "V2": nrm(BOBOT_V2["A_budaya"] * A_C + BOBOT_V2["B_terjemahan"] * B_T)}
assert all(v.shape == Tiv.shape for v in VAR.values())
print({k: v.shape for k, v in VAR.items()})

## Hasil: empat arah, semua varian (val, seed 42)
Baris pertama = baseline M1 (V0). Arah Inggris tidak dipengaruhi varian (angkanya sama dengan M1).

In [ ]:
RANKS, rows = {}, []
for v, Tv in VAR.items():
    for a, f in ARAH.items():
        S = f(Ev, Tv, Tev); m, r = metrics(S); RANKS[(v, a)] = r
        rows.append({"Varian": v, "Arah": a, **m, "query dgn tie": n_tie(S)})
hasil = pd.DataFrame(rows)
hasil.round(4).to_csv(f"{OUT}/hasil/m2_hasil_empat_arah.csv", index=False)
r1 = lambda v, a: float((RANKS[(v, a)] <= 1).mean())

# komit V0 identik dengan M1
p = f"{OUT}/hasil/val_empat_arah.csv"
if os.path.exists(p):
    m1 = pd.read_csv(p, index_col=0)
    for a in ARAH: assert abs(m1.loc[a, "R@1"] - r1("V0", a)) < 5e-4, f"V0 berbeda dari M1 pada {a}"
    print("✔ V0 identik dengan baseline M1 (hasil/val_empat_arah.csv)")

print(f"N = {N} | acak R@1 = {1/N:.4f} | mAP@{K_MAP} acak ≈ {sum(1/i for i in range(1, K_MAP+1))/N:.4f}\n")
for a in ARAH:
    t = hasil[hasil["Arah"] == a].set_index("Varian").drop(columns="Arah")
    print(f"=== {a} ==="); display(t.round(3))

print("\n--- TABEL RINGKAS (kolom template laporan; mAP@%d = arah teks ID → gambar) ---" % K_MAP)
print("| Varian | ID→G R@1 | ID→G R@10 | G→ID R@1 | G→EN R@1 | mAP@%d |" % K_MAP); print("|---|---|---|---|---|---|")
for v in VAR:
    m = hasil[(hasil.Varian == v) & (hasil.Arah == "teks ID → gambar")].iloc[0]
    print(f"| {v} | {r1(v,'teks ID → gambar'):.3f} | {m['R@10']:.3f} | {r1(v,'gambar → teks ID'):.3f} | {r1(v,'gambar → teks EN'):.3f} | {m[f'mAP@{K_MAP}']:.3f} |")

## Language gap per varian
Gap R@1 = R@1 Inggris − R@1 Indonesia (definisi proposal). Sisi Inggris tetap, jadi menyempitnya gap sama dengan naiknya R@1 Indonesia.

In [ ]:
rows = []
for v in VAR:
    gt = r1(v, "gambar → teks EN") - r1(v, "gambar → teks ID")
    gg = r1(v, "teks EN → gambar") - r1(v, "teks ID → gambar")
    rows.append({"Varian": v, "R@1 ID (g→t)": r1(v, "gambar → teks ID"), "gap g→t": gt,
                 "R@1 ID (t→g)": r1(v, "teks ID → gambar"), "gap t→g": gg})
gaptab = pd.DataFrame(rows).set_index("Varian")
gaptab["menyempit g→t (%)"] = (gaptab.loc["V0", "gap g→t"] - gaptab["gap g→t"]) / gaptab.loc["V0", "gap g→t"] * 100
gaptab["menyempit t→g (%)"] = (gaptab.loc["V0", "gap t→g"] - gaptab["gap t→g"]) / gaptab.loc["V0", "gap t→g"] * 100
gaptab.round(4).to_csv(f"{OUT}/hasil/m2_language_gap.csv"); display(gaptab.round(3))

## Apakah selisihnya nyata? Bootstrap berpasangan 95%
Satu seed tidak cukup. Varian di sini **tidak dilatih** (deterministik), jadi ketidakpastiannya berasal dari pengambilan sampel query, bukan seed. Resample query yang **sama** untuk kedua varian (1000 resample, seed 42). Selisih dinyatakan *meyakinkan* hanya bila CI 95% tidak memuat 0.

In [ ]:
m_r1 = lambda r: (r <= 1).astype(float)
m_ap = lambda r: np.where(r <= K_MAP, 1.0 / r, 0.0)

def boot_delta(ra, rb, metric, B=N_BOOT, seed=SEED):
    d = metric(ra) - metric(rb)
    idx = np.random.default_rng(seed).integers(0, len(d), (B, len(d)))     # indeks sama untuk semua pasangan
    lo, hi = np.percentile(d[idx].mean(1), [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi)

PAIRS = [("V1", "V0"), ("V2a", "V0"), ("V2b", "V0"), ("V2", "V0"), ("V2", "V1"), ("V2", "V2a"), ("V2", "V2b")]
rows = []
for a in ARAH_ID:
    for (x, y) in PAIRS:
        for nm, mt in (("R@1", m_r1), (f"mAP@{K_MAP}", m_ap)):
            d, lo, hi = boot_delta(RANKS[(x, a)], RANKS[(y, a)], mt)
            rows.append({"Arah": a, "Perbandingan": f"{x} − {y}", "Metrik": nm, "Selisih": d, "CI95 bawah": lo, "CI95 atas": hi,
                         "Meyakinkan?": "ya" if (lo > 0 or hi < 0) else "belum meyakinkan"})
boot = pd.DataFrame(rows); boot.round(4).to_csv(f"{OUT}/hasil/m2_bootstrap_berpasangan.csv", index=False)
display(boot.round(3))

a = "teks ID → gambar"   # query Indonesia = teks ID → gambar (primer); gambar → teks ID sebagai pembanding
d1 = r1("V1", a) - r1("V0", a); d2 = r1("V2", a) - r1("V1", a)
print(f"\nPengecekan hipotesis proposal M0 (arah {a}, R@1):")
print(f"  H-V1 : |V1 − V0| ≤ 0,02  →  selisih = {d1:+.3f}  →  {'TERPENUHI' if abs(d1) <= 0.02 else 'TIDAK terpenuhi'}")
print(f"  H-V2 : V2 − V1 ≥ 0,05    →  selisih = {d2:+.3f}  →  {'TERPENUHI' if d2 >= 0.05 else 'TIDAK terpenuhi'} (titik); CI di tabel di atas")

## Ablation: efek tiap komponen
Faktorial 2×2: komponen A (templat budaya) dan B (terjemahan), dengan atau tanpa. V1 (templat netral) sebagai kontrol: apakah kenaikan berasal dari *konteks budaya* atau sekadar *efek ensemble*.

In [ ]:
CFG = [("V0", "tidak", "tidak"), ("V1 (kontrol: netral)", "tidak (netral)", "tidak"), ("V2a", "ya", "tidak"), ("V2b", "tidak", "ya"), ("V2", "ya", "ya")]
KEY = {"V0": "V0", "V1 (kontrol: netral)": "V1", "V2a": "V2a", "V2b": "V2b", "V2": "V2"}
rows = []
for nama, A_, B_ in CFG:
    v = KEY[nama]
    rows.append({"Konfigurasi": nama, "Komponen A (templat budaya)": A_, "Komponen B (terjemahan)": B_,
                 "R@1 (ID→G)": r1(v, "teks ID → gambar"), "Selisih vs V0 (ID→G)": r1(v, "teks ID → gambar") - r1("V0", "teks ID → gambar"),
                 "R@1 (G→ID)": r1(v, "gambar → teks ID"), "Selisih vs V0 (G→ID)": r1(v, "gambar → teks ID") - r1("V0", "gambar → teks ID")})
abl = pd.DataFrame(rows); abl.round(4).to_csv(f"{OUT}/hasil/m2_ablation.csv", index=False); display(abl.round(3))

# interaksi A×B: apakah gabungan lebih dari jumlah efek masing-masing?  I = h(V2) − h(V2a) − h(V2b) + h(V0)
for a in ARAH_ID:
    I = m_r1(RANKS[("V2", a)]) - m_r1(RANKS[("V2a", a)]) - m_r1(RANKS[("V2b", a)]) + m_r1(RANKS[("V0", a)])
    idx = np.random.default_rng(SEED).integers(0, N, (N_BOOT, N)); lo, hi = np.percentile(I[idx].mean(1), [2.5, 97.5])
    print(f"Interaksi A×B pada {a}: {I.mean():+.3f}  CI95 [{lo:+.3f}, {hi:+.3f}]  → {'ada sinergi/antagonis' if (lo > 0 or hi < 0) else 'belum meyakinkan (efek A dan B kira-kira aditif)'}")

## Sensitivitas templat (pengganti variasi antar-seed)
Tidak ada seed pelatihan, jadi variasi diukur dengan *leave-one-template-out*: ulangi ensemble dengan membuang tiap templat satu per satu. Ini hanya laporan kekokohan; **tidak** dipakai untuk memilih atau mengganti templat.

In [ ]:
def r1_of(T, a): return float((ranks_of_truth(ARAH[a](Ev, T, Tev)) <= 1).mean())

def loo(PT, B=None):
    out = {a: [] for a in ARAH_ID}
    for k in range(PT.shape[0]):
        A = nrm(np.delete(PT, k, axis=0).mean(0))
        T = A if B is None else nrm(BOBOT_V2["A_budaya"] * A + BOBOT_V2["B_terjemahan"] * B)
        for a in ARAH_ID: out[a].append(r1_of(T, a))
    return {a: np.array(v) for a, v in out.items()}

rows = []
for nama, PT, B in (("V1 (netral)", PT_N, None), ("V2a (budaya)", PT_C, None), ("V2 (budaya+terjemahan)", PT_C, B_T)):
    L = loo(PT, B); v = {"V1 (netral)": "V1", "V2a (budaya)": "V2a", "V2 (budaya+terjemahan)": "V2"}[nama]
    for a in ARAH_ID:
        rows.append({"Konfigurasi": nama, "Arah": a, "R@1 (semua templat)": r1(v, a), "LOO rerata": L[a].mean(), "LOO simpangan baku": L[a].std(ddof=1),
                     "LOO min": L[a].min(), "LOO maks": L[a].max()})
loo_tab = pd.DataFrame(rows); loo_tab.round(4).to_csv(f"{OUT}/hasil/m2_sensitivitas_templat.csv", index=False); display(loo_tab.round(4))
print("Bandingkan simpangan baku LOO dengan selisih antarvarian: bila selisih ≈ simpangan baku, tulis 'belum meyakinkan'.")

## Biaya
Waktu komputasi (CPU, batch, hanya val), jumlah parameter yang **dilatih** (nol di semua varian), dan jumlah parameter inferensi.

In [ ]:
def nparam(m):
    try: return int(sum(p.numel() for p in m.parameters()))
    except Exception: return None

t_v0 = T1.get("val_txt_id")
def tsum(*keys):                                       # None bila ada waktu yang tidak tercatat (bukan 0 palsu)
    v = [TIMES.get(k) for k in keys]
    return None if any(x is None for x in v) else round(sum(v), 1)
cost = pd.DataFrame([
    {"Varian": "V0", "Waktu (dtk)": t_v0, "Encode teks per caption": 1, "Penerjemah saat query": "tidak"},
    {"Varian": "V1", "Waktu (dtk)": tsum("t_netral"), "Encode teks per caption": len(NETRAL), "Penerjemah saat query": "tidak"},
    {"Varian": "V2a", "Waktu (dtk)": tsum("t_budaya"), "Encode teks per caption": len(BUDAYA), "Penerjemah saat query": "tidak"},
    {"Varian": "V2b", "Waktu (dtk)": tsum("t_mt_val", "t_terj_enc"), "Encode teks per caption": 1, "Penerjemah saat query": "ya (NLLB 600M)"},
    {"Varian": "V2", "Waktu (dtk)": tsum("t_budaya", "t_mt_val", "t_terj_enc"), "Encode teks per caption": len(BUDAYA) + 1, "Penerjemah saat query": "ya (NLLB 600M)"},
])
cost["Per caption (ms)"] = cost["Waktu (dtk)"].astype(float) / N * 1000
cost["Parameter dilatih"] = 0
cost.round(2).to_csv(f"{OUT}/hasil/m2_biaya.csv", index=False); display(cost.round(2))
print("Parameter inferensi (beku, tidak dilatih):")
print("  encoder gambar+teks EN :", nparam(img_model))
print("  encoder teks ID        :", nparam(txt_model))
print("  NLLB-200 distilled 600M:", META.get("mt_params"))
print("Catatan waktu: diambil dari cache/waktu_m2.json (tercatat pada eksekusi dengan cache kosong). Sel bernilai None = belum pernah dihitung; hapus cache/m2_* dan hasil/m2_terjemahan_val.csv lalu jalankan ulang untuk mengukurnya.")

## Audit leakage (peran 1)
Varian tidak dilatih, jadi tidak ada gradien yang menyentuh val. Yang diperiksa: (1) konfigurasi terkunci sebelum skor; (2) masukan terjemahan hanya `caption_id`; (3) tumpang tindih val dengan train (caption identik, near-duplicate gambar) dan duplikat caption di dalam val; (4) apakah kesimpulan berubah pada **val bersih** (tanpa baris-baris itu).

In [ ]:
print("1) Konfigurasi terkunci:", json.load(open(LOCK))["waktu_kunci"], "| sha256", SHA[:12])
print("2) Masukan terjemahan = caption_id saja:", cap_id == pd.read_csv(TR_CSV, keep_default_na=False)["caption_id"].astype(str).tolist(),
      "| `category` dan `caption_en` tidak menjadi masukan varian mana pun")

norm = lambda s: re.sub(r"\W+", " ", str(s).lower()).strip()
set_tr_id, set_tr_en = set(map(norm, tr["caption_id"])), set(map(norm, tr["caption_en"]))
dup_tr_id = np.array([norm(c) in set_tr_id for c in cap_id])
dup_tr_en = np.array([norm(c) in set_tr_en for c in va["caption_en"]])
from collections import Counter
cnt = Counter(map(norm, cap_id)); dup_val = np.array([cnt[norm(c)] > 1 for c in cap_id])
dup_img = np.zeros(N, bool)
p_tr = f"{OUT}/cache/train_img.npy"
if os.path.exists(p_tr):
    mx = (Ev @ np.load(p_tr).T).max(1)
    for th in (0.95, 0.98, 0.99): print(f"   gambar val dengan kosinus ke gambar train ≥ {th}: {(mx >= th).sum()}")
    dup_img = mx >= 0.98
else:
    print("   (cache/train_img.npy tidak ada: cek near-duplicate berbasis embedding dilewati)")
print(f"3) caption ID val identik dgn train: {dup_tr_id.sum()} | caption EN: {dup_tr_en.sum()} | caption duplikat di dalam val: {dup_val.sum()} | near-dup gambar lintas split (kos ≥ 0,98): {dup_img.sum()}")

keep = ~(dup_tr_id | dup_tr_en | dup_val | dup_img); print(f"   val bersih: {keep.sum()} dari {N} baris dipertahankan (baris & galeri sama-sama difilter)")
rows = []
for v, Tv in VAR.items():
    for a in ARAH_ID:
        S = ARAH[a](Ev, Tv, Tev)
        rows.append({"Varian": v, "Arah": a, "R@1 val penuh": r1(v, a), "R@1 val bersih": float((ranks_of_truth(S[np.ix_(keep, keep)]) <= 1).mean())})
bersih = pd.DataFrame(rows); bersih["selisih"] = bersih["R@1 val bersih"] - bersih["R@1 val penuh"]
bersih.round(4).to_csv(f"{OUT}/hasil/m2_audit_val_bersih.csv", index=False); display(bersih.round(3))
print("4) Urutan varian dan arah selisih dibandingkan val penuh vs bersih: jika konsisten, tulis 'kesimpulan tidak bergantung pada tumpang tindih'.")

## Analisis: contoh nyata (setelah semua konfigurasi dikunci)
Bagian ini hanya untuk membaca hasil. **Jangan ubah templat lalu jalankan ulang di val** berdasarkan temuan di sini (itu menyetel dengan melihat val).

In [ ]:
a = "teks ID → gambar"
r0, r2, r2b = RANKS[("V0", a)], RANKS[("V2", a)], RANKS[("V2b", a)]
h0, h2 = r0 <= 1, r2 <= 1
print(f"V0 → V2 (R@1, {a}): diperbaiki {int((~h0 & h2).sum())} | dirusak {int((h0 & ~h2).sum())} | benar di keduanya {int((h0 & h2).sum())} | salah di keduanya {int((~h0 & ~h2).sum())}")
print(f"Median peringkat jawaban benar: V0 {np.median(r0):.0f} → V2 {np.median(r2):.0f}")

df = pd.DataFrame({"caption_id": cap_id, "terjemahan_NLLB": trans, "kategori": va["category"], "peringkat V0": r0, "peringkat V2": r2, "peringkat V2b": r2b})
pd.set_option("display.max_colwidth", 110)
print("\n--- 5 query yang paling membaik (V0 → V2) ---")
display(df.assign(naik=r0 - r2).sort_values("naik", ascending=False).head(5).drop(columns="naik"))
print("--- 5 query yang paling memburuk atau tetap gagal (V2) ---")
display(df.assign(turun=r2 - r0).sort_values(["turun", "peringkat V2"], ascending=False).head(5).drop(columns="turun"))

# diagnostik terjemahan: kemiripan kata dengan caption_en bawaan data (TIDAK dipakai oleh varian mana pun)
jac = lambda s, t: len(set(norm(s).split()) & set(norm(t).split())) / max(1, len(set(norm(s).split()) | set(norm(t).split())))
sim = np.array([jac(t, e) for t, e in zip(trans, va["caption_en"])])
print(f"\nDiagnostik: kemiripan kata terjemahan NLLB vs caption_en bawaan: rerata {sim.mean():.3f} | rerata pada query yang benar di V2b {sim[r2b <= 1].mean():.3f} vs salah {sim[r2b > 1].mean():.3f}")

kat = pd.DataFrame({"kategori": va["category"], "d_R1": h2.astype(float) - h0.astype(float)}).groupby("kategori").agg(n=("d_R1", "size"), delta_R1=("d_R1", "mean"))
print("Selisih R@1 (V2 − V0) per kategori (n < 10 tidak stabil, jangan dijadikan klaim):"); display(kat.sort_values("n", ascending=False).round(3))

## Ringkasan, perangkat, dan waktu jalan

In [ ]:
try:
    import transformers; tv = transformers.__version__
except Exception: tv = "?"
print("Perangkat   :", platform.platform(), "|", platform.processor() or "CPU", "|", os.cpu_count(), "core | GPU:", torch.cuda.is_available())
print("Perangkat lunak: Python", sys.version.split()[0], "| numpy", np.__version__, "| torch", torch.__version__, "| sentence-transformers", sentence_transformers.__version__, "| transformers", tv)
print("Seed", SEED, "| bootstrap", N_BOOT, "resample | batch gambar", BATCH_IMG, "| batch teks", BATCH_TXT, "| batch MT", BATCH_MT, "| konfigurasi sha256", SHA[:12])
print("Waktu M2 (detik, cache kosong):", TIMES, "| total:", round(sum(TIMES.values()), 1))

print("\n--- TABEL ABLATION UNTUK LAPORAN (val, seed", SEED, ") ---")
print("| Konfigurasi | Komponen A | Komponen B | R@1 (ID→G) | Selisih vs baseline |"); print("|---|---|---|---|---|")
for _, r in abl.iterrows():
    print(f"| {r['Konfigurasi']} | {r['Komponen A (templat budaya)']} | {r['Komponen B (terjemahan)']} | {r['R@1 (ID→G)']:.3f} | {r['Selisih vs V0 (ID→G)']:+.3f} |")
print("\n--- TABEL BIAYA UNTUK LAPORAN ---")
print("| Varian | Waktu komputasi | Parameter dilatih | Kompleksitas penerapan |"); print("|---|---|---|---|")
for _, r in cost.iterrows():
    print(f"| {r['Varian']} | {r['Waktu (dtk)']} dtk ({r['Per caption (ms)']:.1f} ms/caption) | 0 | {r['Encode teks per caption']}× encode teks; penerjemah saat query: {r['Penerjemah saat query']} |")